# Projet DermaMNIST · Sujet 3 : un outil de tri pour le médecin généraliste

**Groupe :** *(vos trois prénoms)*

## Sujet

DermaMNIST rassemble 10 015 images dermatoscopiques de lésions pigmentées de la peau (issues de la base HAM10000), réparties en 7 types et découpées en train (7 007 images), validation (1 003) et test (2 005). Un médecin généraliste photographie une lésion et doit décider s'il adresse son patient au dermatologue.

**Mission.** Les 7 types sont regroupés en deux catégories (déjà fait dans la cellule de données) : **« à adresser »** (mélanome, carcinome basocellulaire, kératose actinique / carcinome intraépithélial) et **« bénigne »** (les autres). Construire un outil qui rate le moins possible de lésions à adresser.

**Approche.** Un **Vision Transformer pré-entraîné** (par exemple un petit ViT de la bibliothèque `timm`), adapté avec **LoRA** (bibliothèque `peft`) par exemple ! Vous pouvez proposer une autre approche.

**Critère de réussite.** La sensibilité (part des lésions à adresser que l'outil détecte) pour un taux de fausses alertes que vous fixez à l'avance et justifiez. Indiquez aussi le nombre de paramètres entraînés, comparé à un fine-tuning total.

**Baseline.**
- La règle triviale « tout est bénin ».
- Le même ViT dont seule la dernière couche est entraînée.

## Consignes

- **Tout faire dans ce notebook.**
- **Sans IA** : pas de Copilot, de ChatGPT ni d'autre assistant. Vous en êtes capables ! Si besoin, venez me voir.
- **À 15 h, vous présenterez vos résultats** :
  1. Présenter l'objectif
  2. La stratégie que vous avez choisie
  3. Expliquer les différentes étapes que vous avez dû faire
  4. Expliquer les problèmes rencontrés
  5. Présenter vos résultats
  6. Les analyser et les critiquer
- **Comparez les performances obtenues à une baseline** (voir le sujet ci-dessus).

## Récupérer les données

La cellule ci-dessous télécharge DermaMNIST en 64×64 pixels (une seule fois), puis le charge en mémoire :

| Variable | Contenu |
|---|---|
| `X_train`, `X_val`, `X_test` | images, tableaux numpy `uint8` de forme `(N, 64, 64, 3)` |
| `y_train`, `y_val`, `y_test` | classe de chaque image, entier de 0 à 6 |
| `CLASSES` | nom de chaque classe |
| `yb_train`, `yb_val`, `yb_test` | 1 si la lésion est à adresser au dermatologue, 0 sinon |

Vos machines n'ont pas de GPU : mettez votre code au point sur un petit sous-échantillon, mesurez le temps d'une époque, puis seulement passez à plus de données.

In [ ]:
%pip install -q numpy matplotlib requests

# ----- Outil de téléchargement (rien à modifier) -----
import os, hashlib
import requests

def telecharger(url, chemin, md5=None, sha256=None):
    """Télécharge url dans chemin, sauf si le fichier est déjà là."""
    if os.path.exists(chemin):
        print(f"Déjà présent : {chemin}")
        return
    os.makedirs(os.path.dirname(chemin), exist_ok=True)
    print(f"Téléchargement de {os.path.basename(chemin)} :", end=" ")
    empreinte = hashlib.md5() if md5 else hashlib.sha256()
    with requests.get(url, stream=True, timeout=60) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        recu, palier = 0, 0
        with open(chemin + ".part", "wb") as f:
            for bloc in r.iter_content(chunk_size=1 << 20):
                f.write(bloc)
                empreinte.update(bloc)
                recu += len(bloc)
                while total and recu * 100 // total >= palier:
                    print(f"{palier} %", end=" ")
                    palier += 10
    print()
    attendu = md5 or sha256
    if attendu and empreinte.hexdigest() != attendu:
        os.remove(chemin + ".part")
        raise RuntimeError("Fichier corrompu : relancez la cellule.")
    os.replace(chemin + ".part", chemin)

# ----- Chargement des données -----

import numpy as np

DATA_DIR = "data"
TAILLE = 64   # résolution des images : 28, 64, 128 ou 224 pixels (plus grand = plus lent)

FICHIERS = {   # nom du fichier et empreinte MD5 publiés par MedMNIST
    28:  ("dermamnist.npz",     "0744692d530f8e62ec473284d019b0c7"),   # 19,7 Mo
    64:  ("dermamnist_64.npz",  "b70a2f5635c6199aeaa28c31d7202e1f"),   # 100 Mo
    128: ("dermamnist_128.npz", "2defd784463fa5243564e855ed717de1"),   # 373 Mo
    224: ("dermamnist_224.npz", "8974907d8e169bef5f5b96bc506ae45d"),   # 1,1 Go
}
nom, md5 = FICHIERS[TAILLE]
chemin = os.path.join(DATA_DIR, nom)
telecharger(f"https://zenodo.org/records/10519652/files/{nom}?download=1", chemin, md5=md5)

donnees = np.load(chemin)
X_train, y_train = donnees["train_images"], donnees["train_labels"].ravel().astype(np.int64)
X_val,   y_val   = donnees["val_images"],   donnees["val_labels"].ravel().astype(np.int64)
X_test,  y_test  = donnees["test_images"],  donnees["test_labels"].ravel().astype(np.int64)

CLASSES = [
    "kératose actinique / carcinome intraépithélial",   # 0
    "carcinome basocellulaire",                         # 1
    "lésion kératosique bénigne",                       # 2
    "dermatofibrome",                                   # 3
    "mélanome",                                         # 4
    "nævus mélanocytaire (grain de beauté)",            # 5
    "lésion vasculaire",                                # 6
]

# Lésions à adresser au dermatologue : 0 (kératose actinique / carcinome intraépithélial),
# 1 (carcinome basocellulaire) et 4 (mélanome). Toutes les autres sont bénignes.
A_ADRESSER = [0, 1, 4]
yb_train = np.isin(y_train, A_ADRESSER).astype(np.int64)
yb_val   = np.isin(y_val,   A_ADRESSER).astype(np.int64)
yb_test  = np.isin(y_test,  A_ADRESSER).astype(np.int64)

In [ ]:
import textwrap
import matplotlib.pyplot as plt

print("Train :", X_train.shape, "| Validation :", X_val.shape, "| Test :", X_test.shape)
print()
print(f"{'Classe':<50}{'train':>7}{'val':>7}{'test':>7}")
for c, nom in enumerate(CLASSES):
    print(f"{c}  {nom:<47}{(y_train == c).sum():>7}{(y_val == c).sum():>7}{(y_test == c).sum():>7}")
print()
for nom_split, yb in [("train", yb_train), ("val", yb_val), ("test", yb_test)]:
    print(f"Lésions à adresser ({nom_split}) : {yb.sum()} sur {len(yb)} ({yb.mean():.1%})")

fig, axes = plt.subplots(1, 7, figsize=(14, 3))
for c, ax in enumerate(axes):
    i = np.flatnonzero(y_train == c)[0]
    ax.imshow(X_train[i])
    ax.set_title("\n".join(textwrap.wrap(CLASSES[c], 18)), fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
N_PAR_CLASSE = 6
fig, axes = plt.subplots(len(CLASSES), N_PAR_CLASSE, figsize=(2 * N_PAR_CLASSE, 2 * len(CLASSES)))
rng = np.random.default_rng(0)
for c in range(len(CLASSES)):
    indices = np.flatnonzero(y_train == c)
    choisis = rng.choice(indices, size=min(N_PAR_CLASSE, len(indices)), replace=False)
    for j in range(N_PAR_CLASSE):
        ax = axes[c, j]
        ax.axis("off")
        if j < len(choisis):
            ax.imshow(X_train[choisis[j]])
    axes[c, 0].set_title("\n".join(textwrap.wrap(f"{c} · {CLASSES[c]}", 30)), fontsize=8, loc="left")
plt.tight_layout()
plt.show()


## Votre travail